# Script 1 — Prepare the analysis data

This notebook introduces the two processed datasets used in the project and checks the exact analysis-ready tables used by the later scripts.

The default route reads the included files. It does not repeat audio extraction or annotation cleaning. A clearly marked optional section at the end reconstructs the analysis-ready call and sequence tables from the included processed CSV files.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# The notebook works when started from either the repository root or code/.
working_directory = Path.cwd().resolve()
candidates = (working_directory, working_directory.parent)
PROJECT_DIR = next(
    (path for path in candidates if (path / "data").is_dir() and (path / "code").is_dir()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Start this notebook from the repository root or the code directory."
    )

ACOUSTIC_DIR = PROJECT_DIR / "data" / "acoustic"
SEQUENCE_DIR = PROJECT_DIR / "data" / "sequence"
RESULTS_DIR = PROJECT_DIR / "results" / "data_preparation"

ACOUSTIC_SOURCE = ACOUSTIC_DIR / "acoustic_calls_processed.csv"
CALL_ORDER = ACOUSTIC_DIR / "call_order_3612.csv"
SEQUENCE_SOURCE = SEQUENCE_DIR / "sequences_processed.csv"
MULTI_CALL_SEQUENCES = SEQUENCE_DIR / "multi_call_sequences_1619.csv"

for path in (ACOUSTIC_SOURCE, CALL_ORDER, SEQUENCE_SOURCE, MULTI_CALL_SEQUENCES):
    if not path.is_file():
        raise FileNotFoundError(f"Required file is missing: {path.relative_to(PROJECT_DIR)}")

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print(f"Project directory: {PROJECT_DIR}")


## 1. Load the included data

`acoustic_calls_processed.csv` and `sequences_processed.csv` preserve the cleaned outputs of the original annotation workflow. The two smaller analysis tables establish the exact row order and inclusion rules used in the distance matrices.

In [ ]:
acoustic_source = pd.read_csv(ACOUSTIC_SOURCE)
calls = pd.read_csv(CALL_ORDER)
sequence_source = pd.read_csv(SEQUENCE_SOURCE)
sequences = pd.read_csv(MULTI_CALL_SEQUENCES)

expected_rows = {
    "processed acoustic rows": (len(acoustic_source), 3_904),
    "unique calls": (len(calls), 3_612),
    "processed sequence rows": (len(sequence_source), 2_042),
    "multi-call sequences": (len(sequences), 1_619),
}
for label, (observed, expected) in expected_rows.items():
    if observed != expected:
        raise ValueError(f"{label}: found {observed:,}; expected {expected:,}")

pd.DataFrame(
    [
        {"table": label, "rows": observed, "expected": expected}
        for label, (observed, expected) in expected_rows.items()
    ]
)


## 2. Acoustic-call data

The acoustic table contains traditional spectral and temporal measurements, MFCC measurements, recording metadata, and social metadata. Exact duplicate rows in the original processed table represent the same extracted WAV file. Retaining the first occurrence gives the 3,612 unique calls used by Script 2.

`call_id` is the matrix index: row *i* of this table corresponds to row and column *i* of every saved acoustic distance matrix.

In [ ]:
required_call_columns = {
    "call_id", "filename", "call_audio_path", "focal ID", "conspecific_ID",
    "stage", "paired_status", "session_number", "pair_id", "sex",
}
missing = sorted(required_call_columns - set(calls.columns))
if missing:
    raise KeyError(f"The call-order table is missing columns: {missing}")

if not calls["call_id"].to_numpy().tolist() == list(range(len(calls))):
    raise ValueError("call_id must run from 0 to 3,611 in matrix order.")
if calls["filename"].duplicated().any():
    raise ValueError("The call-order table contains duplicate filenames.")
if calls["call_audio_path"].duplicated().any():
    raise ValueError("The call-order table contains duplicate audio paths.")
if set(calls["stage"].dropna()) != {"before", "after"}:
    raise ValueError("Unexpected stage labels in the acoustic table.")
if set(calls["paired_status"].dropna()) != {"partner", "non-partner"}:
    raise ValueError("Unexpected social-context labels in the acoustic table.")

call_counts = (
    calls.groupby(["focal ID", "stage", "paired_status"], observed=True)
    .size()
    .rename("calls")
    .reset_index()
)
display(call_counts.pivot_table(
    index="focal ID",
    columns=["stage", "paired_status"],
    values="calls",
    fill_value=0,
))
display(calls[[
    "call_id", "filename", "focal ID", "conspecific_ID",
    "stage", "paired_status", "session_number",
]].head())


## 3. Sequence data

The processed sequence table contains all retained annotated sequences. The analysis table applies one additional rule: only sequences with at least two recognised call elements are used in the sequence-space analysis.

The accepted call-element alphabet is `A B C D E F K O P R S T`. Boundary markers and text produced by missing spreadsheet cells are not call elements.

In [ ]:
required_sequence_columns = {
    "sequence_id", "source_row", "focal ID", "conspecific_ID", "stage",
    "session_number", "paired_status", "true_sequence", "sequence_length",
}
missing = sorted(required_sequence_columns - set(sequences.columns))
if missing:
    raise KeyError(f"The multi-call sequence table is missing columns: {missing}")

if not sequences["sequence_id"].to_numpy().tolist() == list(range(len(sequences))):
    raise ValueError("sequence_id must run from 0 to 1,618.")
if (sequences["sequence_length"] < 2).any():
    raise ValueError("The analysis table contains a single-element sequence.")

valid_tokens = set("ABCDEFKOPRST")
observed_tokens = set("".join(sequences["true_sequence"].astype(str)))
unexpected_tokens = sorted(observed_tokens - valid_tokens)
if unexpected_tokens:
    raise ValueError(f"Unexpected call elements: {unexpected_tokens}")

sequence_counts = (
    sequences.groupby(["focal ID", "stage", "paired_status"], observed=True)
    .size()
    .rename("sequences")
    .reset_index()
)
display(sequence_counts.pivot_table(
    index="focal ID",
    columns=["stage", "paired_status"],
    values="sequences",
    fill_value=0,
))
display(sequences[[
    "sequence_id", "focal ID", "conspecific_ID", "stage",
    "paired_status", "session_number", "true_sequence",
]].head())


## 4. Shared experimental labels

The acoustic and sequence analyses use the same six focal animals, three established pairs, two recording stages, and two social contexts. Checking these labels here prevents later scripts from silently analysing incompatible subsets.

In [ ]:
def observed_values(frame, column):
    return set(frame[column].dropna().astype(str))

for column in ("focal ID", "stage", "paired_status"):
    acoustic_values = observed_values(calls, column)
    sequence_values = observed_values(sequences, column)
    if acoustic_values != sequence_values:
        raise ValueError(
            f"{column} differs between datasets: "
            f"acoustic={sorted(acoustic_values)}, sequence={sorted(sequence_values)}"
        )

pair_members = {
    "Tabor-Lola": {"Tabor", "Lola"},
    "Odin-Nougatti": {"Odin", "Nougatti"},
    "Wuschel-Olympia": {"Wuschel", "Olympia"},
}
observed_pairs = {
    pair: set(calls.loc[calls["pair_id"] == pair, "focal ID"])
    for pair in pair_members
}
if observed_pairs != pair_members:
    raise ValueError(f"Pair membership does not match the study design: {observed_pairs}")

print("Focal animals:", ", ".join(sorted(calls["focal ID"].unique())))
print("Stages:", ", ".join(sorted(calls["stage"].unique())))
print("Contexts:", ", ".join(sorted(calls["paired_status"].unique())))
print("All shared labels passed.")


## 5. Optional reconstruction of the analysis tables

This section is disabled during an ordinary **Run All**. It reconstructs the two matrix-order tables from the included processed CSV files and writes them to `results/data_preparation/`; it never overwrites the accepted inputs.

Reconstructing the acoustic table also audits all 3,612 WAV files. Install the optional complete-audio archive at `data/acoustic/all_calls/` before enabling it.

In [ ]:
REBUILD_ANALYSIS_TABLES = False

if REBUILD_ANALYSIS_TABLES:
    import soundfile as sf

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    pair_for_individual = {
        "Tabor": "Tabor-Lola",
        "Lola": "Tabor-Lola",
        "Odin": "Odin-Nougatti",
        "Nougatti": "Odin-Nougatti",
        "Wuschel": "Wuschel-Olympia",
        "Olympia": "Wuschel-Olympia",
    }
    sex_for_individual = {
        "Tabor": "male", "Odin": "male", "Wuschel": "male",
        "Lola": "female", "Nougatti": "female", "Olympia": "female",
    }

    rebuilt_calls = acoustic_source.copy()
    rebuilt_calls.insert(
        0, "source_data_row", np.arange(1, len(rebuilt_calls) + 1, dtype=int)
    )
    rebuilt_calls = (
        rebuilt_calls.dropna(subset=["call_audio_path"])
        .drop_duplicates("call_audio_path", keep="first")
        .reset_index(drop=True)
    )
    rebuilt_calls.insert(0, "call_id", np.arange(len(rebuilt_calls), dtype=int))
    rebuilt_calls["pair_id"] = rebuilt_calls["focal ID"].map(pair_for_individual)
    rebuilt_calls["sex"] = rebuilt_calls["focal ID"].map(sex_for_individual)

    if len(rebuilt_calls) != 3_612:
        raise ValueError(
            f"Reconstruction produced {len(rebuilt_calls):,} calls; expected 3,612."
        )

    # The public paths are relative, so the folder can be moved or cloned anywhere.
    rebuilt_calls["call_audio_path"] = rebuilt_calls["filename"].map(
        lambda name: f"data/acoustic/all_calls/{name}"
    )
    missing_audio = [
        path for path in rebuilt_calls["call_audio_path"]
        if not (PROJECT_DIR / path).is_file()
    ]
    if missing_audio:
        raise FileNotFoundError(
            "The optional complete-audio archive is not installed. "
            f"Missing {len(missing_audio):,} WAV files; first path: {missing_audio[0]}"
        )

    # A compact format audit confirms that the matrix order points to the intended audio.
    audio_info = [
        sf.info(PROJECT_DIR / relative_path)
        for relative_path in rebuilt_calls["call_audio_path"]
    ]
    rebuilt_calls["audio_frames"] = [info.frames for info in audio_info]
    rebuilt_calls["duration_s"] = [info.duration for info in audio_info]
    rebuilt_calls["sample_rate_hz"] = [info.samplerate for info in audio_info]
    rebuilt_calls["audio_channels"] = [info.channels for info in audio_info]
    rebuilt_calls["audio_subtype"] = [info.subtype for info in audio_info]

    if not rebuilt_calls["sample_rate_hz"].eq(125_000).all():
        raise ValueError("Not all reconstructed calls use the expected 125-kHz rate.")
    if not rebuilt_calls["audio_channels"].eq(1).all():
        raise ValueError("Not all reconstructed calls are mono.")

    rebuilt_call_path = RESULTS_DIR / "call_order_3612_rebuilt.csv"
    rebuilt_calls.to_csv(rebuilt_call_path, index=False)

    valid_tokens = set("ABCDEFKOPRST")
    rebuilt_sequences = sequence_source.copy()
    rebuilt_sequences.insert(
        0, "source_row", np.arange(1, len(rebuilt_sequences) + 1, dtype=int)
    )
    rebuilt_sequences["true_sequence_source"] = rebuilt_sequences["true_sequence"]
    rebuilt_sequences["true_sequence"] = rebuilt_sequences["true_sequence"].map(
        lambda value: "" if pd.isna(value)
        else "".join(character for character in str(value) if character in valid_tokens)
    )
    rebuilt_sequences["sequence_length"] = (
        rebuilt_sequences["true_sequence"].str.len().astype(int)
    )
    rebuilt_sequences = (
        rebuilt_sequences.loc[rebuilt_sequences["sequence_length"] >= 2]
        .reset_index(drop=True)
    )
    rebuilt_sequences.insert(
        0, "sequence_id", np.arange(len(rebuilt_sequences), dtype=int)
    )

    if len(rebuilt_sequences) != 1_619:
        raise ValueError(
            f"Reconstruction produced {len(rebuilt_sequences):,} sequences; "
            "expected 1,619."
        )

    rebuilt_sequence_path = RESULTS_DIR / "multi_call_sequences_1619_rebuilt.csv"
    rebuilt_sequences.to_csv(rebuilt_sequence_path, index=False)

    print(f"Saved {rebuilt_call_path.relative_to(PROJECT_DIR)}")
    print(f"Saved {rebuilt_sequence_path.relative_to(PROJECT_DIR)}")
else:
    print(
        "Using the included analysis tables. "
        "Set REBUILD_ANALYSIS_TABLES = True only to audit their construction."
    )


## Next step

Script 2 reads `call_order_3612.csv` in this exact order and recreates the four acoustic-space figures from the included distance matrices, fixed embeddings, and example calls.